# Imperial College London - ML/AI Course - May 2026

## Capstone Project - Final Code Notebook

Author: Ian MacKinnon

Date: 3-May-2026


### Overview

This script generates predictions for the ICL ML/AI Capstone BBO Challenge.

It uses Bayesian Optimisation with a UCB or EI acquisition function to select the next sample point for each function

### To use this script

1. Place the numpy input and output files containing this week's data in a /data folder

- the input variable file must be called **function**_**week**_inputs.npy e.g. f2_w6_inputs.npy
- the output variable file must be called **function**_**week**_outputs.npy e.g. f2_w6_outputs.npy

2. In the global variables:

- set the **week** variable to be this week of the capstone project
- set the **functionID** variable to be the value of the function number (1 through 8) for which you wish to generate a sample point
- ensure the path in the **xfile** and **yfile** variables points to your /data folder
- set the **verbose** variable to true if you wish to examine the winning candidate number and expected result

3. in the function specific variables:

- set the **kernel** type and hyperparameters you wish to use (e.g. ConstantKernel, RBF, Matern, length_scale, nu)
- set the type of **acquisition function** you wish to use
- set the hyperparameters of the acquisition function (Kappa or Xi)

### How it works

The script will:
- load the numpy files into arrays
- build a gaussian process surrogate of the true function
- run an aquisition function to:
    - generate a large number of candidate points
    - process them using the gaussian process (surrogate function)
    - select the best candidate point based on the acquisition function hyperparameters
    - format and print that sample point


### First, load the required libraries

In [12]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Matern, WhiteKernel
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt

### Edit the global variables

- the **week** parameter must be set to the week for which you wish to predict. For example, if you wish to predict for Week 4, set week = 4
- the **functionID** parameter must be set to the number of the function you wish to predict. For example, if you wish to predict F7, set functionID = 7
- optionally, the **verbose** parameter can be set to True or False. Set it to true if you want console output and the forecast value of the recommended sample point

In [13]:
week = 4
functionID = 7
verbose = False
np.random.seed(42)

### Check the path to the data files

Only edit this section of the filepath **C:/users/ianma/module0_capstone** - the remainder of the filename will be derived from the global variables above

In [14]:
xfile = "C:/users/ianma/module0_capstone/week_" + str(week) + "/data/f" + str(functionID) + "_w" + str(week) + "_inputs.npy"
yfile = "C:/users/ianma/module0_capstone/week_" + str(week) + "/data/f" + str(functionID) + "_w" + str(week) + "_outputs.npy"

### Edit the function-specific parameters

These include the kernel and acquisition function:

- **Kernel** can ConstantKernel, RBF, Matern or Whitekernel or a combination thereof
- Acquisition function **aqFunc** can be UCB or EI
- **Kappa** controls the UCB acquisition function. A higher value of Kappa will explore while a lower value will exploit
- **Xi** controls the EI acquisition function. A higher value of xi will explore while a lower value will exploit

In [15]:
if functionID == 1:
    #kernel = 1.0 * Matern(length_scale=1.0, nu=1.5) # Matern kernel encourages even more exploitation
    kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)    
    aqFunc = 'ucb'
    kappa = 0.001 # switched to UCB and dropped from 20 to 0.1 for wk7 to encourage exploitation
    xi = 0.1 
    bounds = np.array([[0, 0], [1, 1]])
    dimensions = 2
#
if functionID == 2:
    #kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
    kernel = 1.0 * Matern(length_scale=0.05, nu=2.5)
    aqFunc = 'ucb'
    kappa = 0.01 # switched to UCB and dropped from 20 to 0.1 for wk7 to encourage exploitation
    xi = 0.01 
    bounds = np.array([[0, 0], [1, 1]])
    dimensions = 2
#
if functionID == 3:
    #kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
    kernel = 1.0 * Matern(length_scale=0.05, nu=2.5)
    aqFunc = 'ucb'
    kappa = 0.01 # switched to UCB and dropped from 20 to 0.1 for wk7 to encourage exploitation
    xi = 0.1
    bounds = np.array([[0, 0, 0], [1, 1, 1]])
    dimensions = 3
#
if functionID == 4:
    #kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
    kernel = 1.0 * Matern(length_scale=0.05, nu=2.5)
    aqFunc = 'ei'
    kappa = 0.01
    xi = 0.01 
    bounds = np.array([[0, 0, 0, 0], [1, 1, 1, 1]])
    dimensions = 4
#
if functionID == 5:
    #kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
    kernel = 1.0 * Matern(length_scale=0.05, nu=2.5)
    aqFunc = 'ei'
    kappa = 0.01 # switched to UCB and dropped from 20 to 0.1 for wk7 to encourage exploitation
    xi = 0.1 
    bounds = np.array([[0, 0, 0, 0], [1, 1, 1, 1]])
    dimensions = 4
#
if functionID == 6:
    #kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
    kernel = 1.0 * Matern(length_scale=0.05, nu=2.5)
    aqFunc = 'ucb'
    kappa = 0.01 # switched to UCB and dropped from 20 to 0.1 for wk7 to encourage exploitation
    xi = 0.01 
    bounds = np.array([[0, 0, 0, 0, 0], [1, 1, 1, 1, 1]])
    dimensions = 5
#
if functionID == 7:
    #kernel = WhiteKernel(noise_level=0.1)
    #kernel = ConstantKernel(1) * RBF(length_scale=0.1)
    #kernel = ConstantKernel(1.0) * WhiteKernel(noise_level=0.1)
    kernel = 1.0 * Matern(length_scale=0.05, nu=2.5)
    aqFunc = 'ucb'
    kappa = 0.01
    xi = 0.01
    bounds = np.array([[0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1]])
    dimensions = 6
#
if functionID == 8:
    #kernel = WhiteKernel(noise_level=0.1)
    #kernel = Matern(length_scale=1.0, nu=1.0) * WhiteKernel(noise_level=0.1)
    #kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
    kernel = 1.0 * Matern(length_scale=0.05, nu=2.5)
    aqFunc = 'ei'
    kappa = 0.01
    xi = 0.01
    bounds = np.array([[0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1]])
    dimensions = 8


### Import the data

In [16]:
X = np.load(xfile)
Y = np.load(yfile)
Y_best = Y.max()


### Fit the Gaussian Process

In [17]:
kernel = ConstantKernel(1.0) * RBF(length_scale=0.1)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gp.fit(X, Y)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",1**2 * RBF(length_scale=0.1)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",10
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",False
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`Gl

### Define the Upper Confidence Bound (UCB) Acquisition Function

In [18]:
def ucbAcquisition(X_cand, gp, kappa):
    mean, std = gp.predict(X_cand, return_std=True)
    return mean + kappa * std  # We want to maximize this value

###  Define the Expected Improvement (EI) Acquisition Function

In [19]:
def eiAcquisition(X_cand, gp, Y_best, xi):
    mean, std = gp.predict(X_cand, return_std=True)
    with np.errstate(divide='warn'):
        improvement = mean - Y_best - xi
        Z = improvement / std
        ei = improvement * norm.cdf(Z) + std * norm.pdf(Z)
        ei[std == 0.0] = 0.0
    return ei

### Define a function to generate and return the recommended sample point

In [20]:
def propose_next_point(gp, bounds, dimensions, n_candidates=10000):
    # Randomly sample candidates in nD space
    candidates = np.random.uniform(bounds[0], bounds[1], size=(n_candidates, dimensions))
    # Call the relevant acquisition function
    if aqFunc == 'ucb':
        scores = ucbAcquisition(candidates, gp, kappa)
        if verbose:
            print('\nAq Function: UCB', '\nY_best:', Y_best, '\nKappa:',  kappa)
    if aqFunc == 'ei':
        if verbose:
            print('\nAq Function: EI', '\nY_best:', Y_best, '\nXi:',  xi)
        scores = eiAcquisition(candidates, gp, Y_best, xi)
        
    # Return the candidate with the highest score
    next_point = candidates[np.argmax(scores)]
    
    if verbose:
        print('Winning candidate number: ', np.argmax(scores))
        print('Next point pre-decimal adjustment:', next_point)

    next_point = np.round(next_point,6)
    return next_point

### Main Function - request the next sample point

In [21]:
next_point = propose_next_point(gp, bounds, dimensions)

### Create and print the query string for submission

In [22]:
columns= len(next_point)
colcount = 0
query=''
packzero = ''
while colcount < columns:
    element = str(round(next_point[colcount], 6))
    strlen = len(element)
    if strlen < 8:
        packlen = 8 - strlen
        while len(packzero) < packlen:
            packzero = packzero + '0'
        element = element + packzero
    query = query + '-' + element
    colcount = colcount + 1

#8. Print the query string
print(f"Function {functionID}")
print (query[1:])

Function 7
0.271651-0.232006-0.399051-0.224999-0.352599-0.728543
